# Analyse Prédictive de la Consommation Énergétique des Logements (DPE)

**Objectif principal :** Ce notebook a pour but d'entraîner, de comparer et d'analyser plusieurs algorithmes de Machine Learning afin de prédire la consommation énergétique d'un logement en fonction de ses caractéristiques physiques (surface, année de construction, isolation, type de chauffage, etc.).

### Déroulement de l'analyse :
1. **Chargement des données :** Importation du jeu de données régional préalablement nettoyé et agrégé (ex: `dpe_IDF.csv`).
2. **Prétraitement (Preprocessing) :** Transformation des variables catégorielles (texte) en données exploitables par les algorithmes (*One-Hot Encoding*) et mise à l'échelle des variables numériques (*StandardScaler*).
3. **Entraînement des modèles (Training) :** Mise en compétition de 4 algorithmes différents :
   - *Modèles linéaires :* Régression Linéaire, Régression Ridge.
   - *Modèles ensemblistes (Arbres) :* Random Forest, XGBoost.
4. **Évaluation des performances :** Comparaison des modèles à l'aide de deux métriques clés :
   - **R² (Coefficient de détermination) :** La précision globale du modèle.
   - **RMSE (Root Mean Squared Error) :** L'erreur moyenne de prédiction.
5. **Analyse des facteurs clés (Feature Importance) :** Identification des variables (features) qui ont le plus d'impact sur la consommation énergétique selon le meilleur modèle.

---
* Note : Ce notebook est conçu pour être modulaire. Il suffit de changer la variable `region_code` dans la première cellule de code pour relancer l'analyse complète sur un autre territoire.*

### Importations

Cette première étape prépare notre environnement de travail. Nous importons Pandas et Numpy pour manipuler les données, Matplotlib et Seaborn pour tracer nos graphiques, et Scikit-Learn / XGBoost pour toute la partie Machine Learning.
Nous fixons aussi le style visuel de nos graphiques.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Modules de Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import xgboost as xgb

# Configuration visuelle des graphiques
sns.set_theme(style="whitegrid", palette="muted")
import warnings
warnings.filterwarnings('ignore') # Masque les petits avertissements inutiles

Ici, nous définissons la région à analyser. Le code vérifie si le fichier CSV généré par vos précédentes étapes de pipeline (ex: dpe_IDF.csv) existe bien sur votre ordinateur. S'il le trouve, il le charge en mémoire (dans un DataFrame appelé `df`).

In [ ]:
# Paramètres de la région (Peut être écrasé par Papermill)
region_code = 'IDF'
fichier_donnees = f'dpe_{region_code}.csv'

print(f"Recherche du fichier : {fichier_donnees}...")

if os.path.exists(fichier_donnees):
    df = pd.read_csv(fichier_donnees)
    print(f"Succès : Dataset chargé avec {df.shape[0]} logements et {df.shape[1]} colonnes.")
else:
    print(f"Erreur critique : Le fichier {fichier_donnees} est introuvable. Avez-vous lancé la fusion ?")

### Définition des Variables et Prétraitement

**Target** : C'est ce qu'on veut deviner (la consommation d'énergie).

**Features (Variables explicatives)** : On sépare les textes (catégories) des nombres. J'ai listé ici toutes les variables courantes du DPE.

**Nettoyage** : On retire les lignes où la cible est vide ou absurde (<= 0).

**Prétraitement (Preprocessor)** : Les algorithmes ne comprennent pas le texte. Le OneHotEncoder va transformer les catégories (ex: "Gaz", "Bois") en colonnes de 0 et de 1. Le StandardScaler va mettre tous les nombres à la même échelle pour aider le modèle.

In [ ]:
# 1. Variable cible (Target)
target = 'consommation_energie' # ou 'consommation_energie_finale', selon votre CSV

# 2. Variables Numériques (Quantitatives)
features_num = [
    'surface_habitable', 
    'annee_construction',
    'hauteur_sous_plafond',
    'surface_toiture',
    'surface_murs_exterieurs',
    'surface_vitree_nord',
    'surface_vitree_sud',
    'surface_vitree_est_ouest'
]

# 3. Variables Catégorielles (Qualitatives / Textes)
features_cat = [
    'chauffage_simplifie', 
    'logement_traversant_clean', 
    'isolation_toiture_clean',
    'type_batiment', # ex: Appartement, Maison
    'periode_construction',
    'type_ventilation',
    'type_production_ecs', # Eau Chaude Sanitaire
    'zone_climatique',
    'tranche_surface'
]



In [ ]:
# --- Filtrage de sécurité ---
# On s'assure que seules les colonnes qui existent VRAIMENT dans le df sont gardées
features_num = [col for col in features_num if col in df.columns]
features_cat = [col for col in features_cat if col in df.columns]

# Nettoyage des valeurs aberrantes sur la cible
df_clean = df.dropna(subset=[target]).copy()
df_clean = df_clean[np.isfinite(df_clean[target])]
df_clean = df_clean[df_clean[target] > 0]

print(f"Logements restants après nettoyage de la cible : {df_clean.shape[0]}")

# Définition X (Features) et y (Target)
X = df_clean[features_cat + features_num]
y = df_clean[target]

# Séparation: 80% pour l'entraînement (Train), 20% pour l'évaluation (Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Création du "Traducteur" de données (Preprocessor)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), features_num),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features_cat)
    ]
)
print("Prétraitement configuré (StandardScaler + OneHotEncoder).")

### Entraînement des Différents Algorithmes

In [ ]:
# Dictionnaire regroupant nos pipelines
models = {
    "Reg Linéaire": Pipeline([('preprocessor', preprocessor), ('regressor', LinearRegression())]),
    "Ridge": Pipeline([('preprocessor', preprocessor), ('regressor', Ridge(alpha=1.0))]),
    "Random Forest": Pipeline([('preprocessor', preprocessor), ('regressor', RandomForestRegressor(n_estimators=50, max_depth=10, random_state=42, n_jobs=-1))]),
    "XGBoost": Pipeline([('preprocessor', preprocessor), ('regressor', xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42))])
}

results = {}

print(f" Début du tournoi des modèles pour {region_code}...\n")

for name, pipeline in models.items():
    print(f"⏳ Entraînement de {name}...")
    
    # Le modèle apprend sur les données Train
    pipeline.fit(X_train, y_train)
    
    # Le modèle passe l'examen sur les données Test
    y_pred = pipeline.predict(X_test)
    
    # Correction de l'examen (Calcul des scores)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # L'erreur moyenne (en kWh)
    r2 = r2_score(y_test, y_pred) # Le % de prédiction correcte (Proche de 1 = Parfait)
    
    # On stocke tout pour l'analyse
    results[name] = {"RMSE": rmse, "R2": r2, "Pipeline": pipeline}
    
    print(f"   ↳ Terminé ! R² = {r2:.3f} | Erreur moy. (RMSE) = {rmse:.0f}\n")

### Analyse des Résultats (Graphes et Variables)

1. Graphes de comparaison : Nous visualisons quel algorithme a le meilleur R² et la plus petite erreur (RMSE).
2. Importance des variables (Feature Importance) : Nous prenons le meilleur modèle et lui demandons "Qu'est-ce qui t'a le plus aidé à deviner la consommation ?". Est-ce l'année de construction ? Le type de chauffage ? Le graphique affiche le Top 30 des variables qui ont le plus de poids dans la décision de l'algorithme.

In [ ]:
# --- 1. Graphiques des scores ---
metrics_df = pd.DataFrame(results).T[['RMSE', 'R2']]

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
metrics_df['R2'].plot(kind='bar', ax=ax[0], color='#2ecc71', title='Comparaison des R² (Le plus haut gagne)')
metrics_df['RMSE'].plot(kind='bar', ax=ax[1], color='#e74c3c', title='Comparaison RMSE (Le plus bas gagne)')
ax[0].set_ylim(0, 1)
plt.tight_layout()
plt.show()

# --- 2. Feature Importance du Meilleur Modèle ---
best_model_name = metrics_df['R2'].astype(float).idxmax()
best_pipeline = results[best_model_name]['Pipeline']
print(f"Le meilleur modèle est : {best_model_name}")

if best_model_name in ["Random Forest", "XGBoost"]:
    
    # Étape un peu technique : récupérer les vrais noms des colonnes après le passage du OneHotEncoder
    cat_encoder = best_pipeline.named_steps['preprocessor'].named_transformers_['cat']
    encoded_cat_names = cat_encoder.get_feature_names_out(features_cat)
    all_feature_names = list(features_num) + list(encoded_cat_names)
    
    # Récupération des poids (importances) donnés par le modèle
    importances = best_pipeline.named_steps['regressor'].feature_importances_
    
    # Création d'un tableau trié
    feat_imp_df = pd.DataFrame({'Feature': all_feature_names, 'Importance': importances})
    feat_imp_df = feat_imp_df.sort_values(by='Importance', ascending=False).reset_index(drop=True)
    
    # Affichage du texte dans la console
    top_30 = feat_imp_df.head(30)
    print("\n=== TOP 30 VARIABLES LES PLUS IMPORTANTES ===")
    for i, (idx, row) in enumerate(top_30.iterrows(), 1):
        print(f"{i}. {row['Feature']:<50} (Importance : {row['Importance']:.2%})")
        
    # Visualisation Magnifique
    plt.figure(figsize=(10, 12))
    sns.barplot(x='Importance', y='Feature', data=top_30, palette='magma')
    plt.title(f'Top 30 Variables par Importance ({best_model_name} - {region_code})')
    plt.xlabel('Importance Relative (Poids dans la décision)')
    plt.ylabel('Variable')
    plt.tight_layout()
    plt.show()
else:
    print(f"\n Le modèle {best_model_name} n'est pas basé sur des arbres, l'importance des variables n'est pas calculée.")